# ДЗ4. Full fine-tuning и LoRA

Qwen2.5-1.5B-Instruct, function calling на xLAM. Сравним исходную модель, Full FT и LoRA, затем по отдельности изменим `r`, `alpha`, `dropout`, `target_modules`.

Запускаем на одном GPU с BF16.

In [1]:
import gc
import json
import re
import time
from pathlib import Path

import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import snapshot_download
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
    set_seed,
)

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
SEED = 42
MAX_LENGTH = 2048
MAX_NEW_TOKENS = 512
OUTPUT = Path("outputs")
OUTPUT.mkdir(exist_ok=True)
set_seed(SEED)
checkpoint = snapshot_download(MODEL)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
tokenizer.pad_token = tokenizer.eos_token


## 1. Данные и loss

xLAM содержит запросы, описания доступных функций и правильные вызовы с аргументами. Это непосредственно наша задача. Приведём tools к JSON Schema, ответы к `tool_calls` родного chat template.

После исключения слишком длинных примеров случайно выберем 11 500: 10 000 train, 500 validation, 1 000 test. Split один для всех моделей.

In [2]:
raw = load_dataset("Salesforce/xlam-function-calling-60k", split="train")
types = {
    "int": "integer", "float": "number", "str": "string",
    "bool": "boolean", "list": "array", "tuple": "array",
    "set": "array", "dict": "object",
}


def chat(row):
    tools = []
    for function in json.loads(row["tools"]):
        properties = {}
        required = []
        for name, spec in function["parameters"].items():
            spec = spec.copy()
            annotation = spec.pop("type")
            is_required = not ("optional" in annotation or "default" in annotation or "default" in spec)
            if spec.pop("required", is_required):
                required.append(name)
            kind = re.split(r"[,\[\s]", annotation.strip().lower(), maxsplit=1)[0]
            kind = types.get(kind, kind)
            if kind in {"string", "integer", "number", "boolean", "array", "object", "null"}:
                spec["type"] = kind
            spec["description"] = f"{spec.get('description', '')} Python type: {annotation}.".strip()
            properties[name] = spec
        function["parameters"] = {
            "type": "object", "properties": properties, "required": required,
        }
        tools.append({"type": "function", "function": function})
    calls = [
        {"type": "function", "function": call}
        for call in json.loads(row["answers"])
    ]
    messages = [
        {"role": "user", "content": row["query"]},
        {"role": "assistant", "content": "", "tool_calls": calls},
    ]
    return tools, messages


def tokenize(row):
    tools, messages = chat(row)
    prompt = tokenizer.apply_chat_template(
        messages[:-1], tools=tools, add_generation_prompt=True,
    )
    full = tokenizer.apply_chat_template(messages, tools=tools)
    assert full[:len(prompt)] == prompt
    return {
        "input_ids": full,
        "attention_mask": [1] * len(full),
        "labels": [-100] * len(prompt) + full[len(prompt):],
    }


encoded = raw.map(tokenize)
encoded = encoded.filter(
    lambda row: len(row["input_ids"]) <= MAX_LENGTH
    and sum(token != -100 for token in row["labels"]) <= MAX_NEW_TOKENS
)
selected = encoded.shuffle(seed=SEED).select(range(11500))
split = selected.train_test_split(test_size=1000, seed=SEED)
train_val = split["train"].train_test_split(test_size=500, seed=SEED)
columns = ["input_ids", "attention_mask", "labels"]
train = train_val["train"].select_columns(columns)
validation = train_val["test"].select_columns(columns)
test = split["test"]
display(pd.DataFrame(test.select(range(2)))[["query", "tools", "answers"]])
print("Train / validation / test:", len(train), len(validation), len(test))
print("Обучаемый текст первого примера:")
print(tokenizer.decode([token for token in train[0]["labels"] if token != -100]))


query  ...                                            answers
0  Can you find the geolocation of IP addresses 8...  ...  [{"name": "ip_geolocation", "arguments": {"ip_...
1  Calculate the median of the list [10, 20, 30, ...  ...  [{"name": "calculate_median", "arguments": {"n...

[2 rows x 3 columns]
Train / validation / test: 10000 500 1000
Обучаемый текст первого примера:
<tool_call>
{"name": "min_meeting_rooms", "arguments": {"intervals": [[0, 10], [2, 8], [10, 15], [12, 17], [15, 20]]}}
</tool_call><|im_end|>


## 2. Две оценки

**Tool exact match:** должны совпасть функции и все аргументы. Пробелы и порядок JSON-ключей не важны; порядок независимых вызовов тоже игнорируем, повторы учитываем. Лишний текст, незавершённый ответ или неверный JSON дают ошибку.

**MMLU:** общие знания и возможное забывание. Используем полный `cais/mmlu/all/test`, zero-shot: выбираем наиболее вероятную букву A/B/C/D после chat prompt. 

In [3]:
mmlu = load_dataset("cais/mmlu", "all", split="test")
letters = [tokenizer.encode(letter, add_special_tokens=False) for letter in "ABCD"]
assert all(len(ids) == 1 for ids in letters)
letter_ids = [ids[0] for ids in letters]


def exact_match(text, expected):
    blocks = re.findall(r"<tool_call>(.*?)</tool_call>", text, flags=re.S)
    if re.sub(r"<tool_call>.*?</tool_call>", "", text, flags=re.S).strip():
        return False
    try:
        predicted = [
            json.dumps(json.loads(block), sort_keys=True, ensure_ascii=False)
            for block in blocks
        ]
        expected = [
            json.dumps(call, sort_keys=True, ensure_ascii=False)
            for call in expected
        ]
        return sorted(predicted) == sorted(expected)
    except (ValueError, TypeError):
        return False


@torch.inference_mode()
def evaluate(model):
    model.eval()
    tokenizer.padding_side = "left"
    correct = 0
    examples = []
    eos = model.generation_config.eos_token_id
    eos = eos if isinstance(eos, list) else [eos]
    for start in tqdm(range(0, len(test), 8), desc="Tool EM"):
        batch = test.select(range(start, min(start + 8, len(test))))
        prompts = []
        for row in batch:
            tools, messages = chat(row)
            prompts.append(tokenizer.apply_chat_template(
                messages[:-1], tools=tools, tokenize=False, add_generation_prompt=True,
            ))
        inputs = tokenizer(
            prompts, padding=True, add_special_tokens=False, return_tensors="pt",
        ).to("cuda")
        output = model.generate(
            **inputs, do_sample=False, max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=tokenizer.pad_token_id,
        )
        answers = output[:, inputs["input_ids"].shape[1]:].tolist()
        for row, answer in zip(batch, answers):
            ended = any(token in eos for token in answer)
            if ended:
                answer = answer[:next(i for i, token in enumerate(answer) if token in eos)]
            text = tokenizer.decode(answer, skip_special_tokens=False)
            expected = json.loads(row["answers"])
            correct += ended and exact_match(text, expected)
            if len(examples) < 10:
                examples.append({
                    "query": row["query"], "expected": expected, "prediction": text,
                })
    tool_accuracy = correct / len(test)

    correct = 0
    for start in tqdm(range(0, len(mmlu), 16), desc="MMLU"):
        batch = mmlu.select(range(start, min(start + 16, len(mmlu))))
        prompts = []
        for row in batch:
            choices = "\n".join(
                f"{letter}. {choice}"
                for letter, choice in zip("ABCD", row["choices"])
            )
            question = f"{row['question']}\n{choices}\nAnswer with only A, B, C or D."
            prompts.append(tokenizer.apply_chat_template(
                [{"role": "user", "content": question}],
                tokenize=False, add_generation_prompt=True,
            ))
        inputs = tokenizer(
            prompts, padding=True, add_special_tokens=False, return_tensors="pt",
        ).to("cuda")
        positions = (inputs["attention_mask"].cumsum(-1) - 1).clamp_min(0)
        logits = model(**inputs, position_ids=positions, logits_to_keep=1).logits[:, -1, letter_ids]
        predicted = logits.argmax(-1).tolist()
        correct += sum(p == row["answer"] for p, row in zip(predicted, batch))
    tokenizer.padding_side = "right"
    return tool_accuracy, correct / len(mmlu), examples


## 3. Исходная модель, Full FT и LoRA

Обучаем одну эпоху с эффективным batch 16 (batch 4, накопление 4) и seed 42. Full FT: learning rate `2e-5`, LoRA: `1e-4`.

Базовая LoRA: `r=16`, `alpha=32`, `dropout=0`, `q_proj+v_proj`. В каждом варианте меняем один параметр. При изменении `r` меняется и масштаб `alpha/r`.

В Full FT храним веса и Adam states в FP32 и используем BF16 autocast. Оцениваем все модели в BF16. Замеряем время `trainer.train()` с validation и пик allocated GPU-памяти.

Оценку считаем пакетами: 8 запросов для tool EM и 16 для MMLU.


In [4]:
lora_default = dict(
    r=16, lora_alpha=32, lora_dropout=0.0, target_modules=["q_proj", "v_proj"],
)
experiments = [
    ("base", None),
    ("full", None),
    ("lora", {}),
    ("r8", {"r": 8}),
    ("alpha16", {"lora_alpha": 16}),
    ("dropout01", {"lora_dropout": 0.1}),
    ("all_linear", {"target_modules": "all-linear"}),
]
results = []
manual = []

for name, change in experiments:
    set_seed(SEED)
    dtype = torch.float32 if name == "full" else torch.bfloat16
    model = AutoModelForCausalLM.from_pretrained(checkpoint, dtype=dtype).to("cuda")
    lora_config = {}
    if change is not None:
        lora_config = lora_default | change
        model = get_peft_model(model, LoraConfig(**lora_config, task_type="CAUSAL_LM"))
    trainable = 0
    seconds = 0.0
    peak = 0.0
    if name != "base":
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        model.config.use_cache = False
        args = TrainingArguments(
            output_dir=str(OUTPUT / name),
            num_train_epochs=1,
            learning_rate=2e-5 if name == "full" else 1e-4,
            per_device_train_batch_size=4,
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=4,
            bf16=True,
            optim="adamw_torch",
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
            eval_strategy="epoch",
            save_strategy="no",
            report_to="none",
            logging_steps=50,
            logging_first_step=True,
            seed=SEED,
        )
        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=train,
            eval_dataset=validation,
            data_collator=DataCollatorForSeq2Seq(tokenizer),
            processing_class=tokenizer,
        )
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        started = time.perf_counter()
        trainer.train()
        torch.cuda.synchronize()
        seconds = time.perf_counter() - started
        peak = torch.cuda.max_memory_allocated() / 2**30
        trainer.save_model(str(OUTPUT / name))
        del trainer
    model.config.use_cache = True
    if lora_config:
        model = model.merge_and_unload()
    model.to(torch.bfloat16)
    tool_em, mmlu_accuracy, examples = evaluate(model)
    results.append({
        "model": name,
        **lora_config,
        "trainable_parameters": trainable,
        "seconds": seconds,
        "peak_gpu_gib": peak,
        "tool_em": tool_em,
        "mmlu": mmlu_accuracy,
    })
    if name in ["base", "full", "lora"]:
        for example in examples:
            manual.append({"model": name, **example})
    pd.DataFrame(results).to_csv(OUTPUT / "results.csv", index=False)
    pd.DataFrame(manual).to_csv(OUTPUT / "manual_review.csv", index=False)
    del model
    gc.collect()
    torch.cuda.empty_cache()

display(pd.DataFrame(results))


{'loss': 0.0389, 'grad_norm': 1.9707183837890625, 'learning_rate': 2e-05, 'epoch': 0.0}

{'loss': 0.0671, 'grad_norm': 0.8276388645172119, 'learning_rate': 1.8432000000000002e-05, 'epoch': 0.08}

{'loss': 0.0535, 'grad_norm': 1.777169108390808, 'learning_rate': 1.6832e-05, 'epoch': 0.16}

{'loss': 0.0744, 'grad_norm': 25.712305068969727, 'learning_rate': 1.5232000000000003e-05, 'epoch': 0.24}

{'loss': 0.054, 'grad_norm': 0.48852869868278503, 'learning_rate': 1.3632000000000001e-05, 'epoch': 0.32}

{'loss': 0.0593, 'grad_norm': 1.5519505739212036, 'learning_rate': 1.2032000000000001e-05, 'epoch': 0.4}

{'loss': 0.0368, 'grad_norm': 0.8180988430976868, 'learning_rate': 1.0432e-05, 'epoch': 0.48}

{'loss': 0.0482, 'grad_norm': 0.31892040371894836, 'learning_rate': 8.832000000000001e-06, 'epoch': 0.56}

{'loss': 0.0397, 'grad_norm': 2.3863658905029297, 'learning_rate': 7.232e-06, 'epoch': 0.64}

{'loss': 0.0427, 'grad_norm': 1.5306438207626343, 'learning_rate': 5.6320000000000005e-06, 'ep

## 4. Результаты

Полный прогон выполнен на одной A100 80 GB. Все семь моделей оценены на одних и тех же 1 000 xLAM-примерах и 14 042 вопросах MMLU.

| Модель | Обучаемых параметров | Время, мин | Пик, GiB | Tool EM, % | MMLU, % |
|---|---:|---:|---:|---:|---:|
| `base` | 0 | - | - | 72.10 | 57.69 |
| `full` | 1 543 714 304 | 17.10 | 38.69 | 78.70 | 53.84 |
| `lora` | 2 179 072 | 13.06 | 17.42 | 77.60 | 57.48 |
| `r8` | 1 089 536 | 13.02 | 17.41 | 78.30 | 57.34 |
| `alpha16` | 2 179 072 | 13.04 | 17.42 | 77.00 | 57.43 |
| `dropout01` | 2 179 072 | 13.28 | 17.42 | 78.00 | 57.45 |
| `all_linear` | 18 464 768 | 18.57 | 17.60 | 79.10 | 56.74 |

Время включает `trainer.train()` и validation, но не сохранение весов и тестирование. Пик памяти измерен через `torch.cuda.max_memory_allocated()` во время обучения.

Full FT поднял tool EM с 72,1% до 78,7%, но MMLU снизился на 3,85 п.п. Базовая LoRA дала 77,6% tool EM и потеряла только 0,21 п.п. MMLU. Она обучала 2,18 млн параметров, около 0,14% размера исходной модели. Пик памяти оказался в 2,22 раза меньше, время обучения сократилось с 17,10 до 13,06 минуты.

Изменения относительно базовой LoRA:

- `r=8`: 78,3% tool EM, вдвое меньше обучаемых параметров. Пик памяти и время почти не изменились. Здесь вместе с рангом изменился масштаб `alpha/r` с 2 до 4.
- `alpha=16`: 77,0% tool EM, улучшения не получилось. MMLU остался близок к базовой LoRA.
- `dropout=0.1`: 78,0% tool EM, на 0,4 п.п. выше базовой LoRA при почти том же MMLU и расходе памяти.
- `all-linear`: лучший tool EM, 79,1%, но MMLU снизился до 56,74%. Обучаемых параметров стало 18,46 млн, время выросло до 18,57 минуты. Этот вариант оказался медленнее Full FT, сохранив заметно меньший пик памяти.

В этом прогоне `all-linear` дал максимум tool EM, а базовая LoRA лучше остальных обученных вариантов сохранила MMLU. `r=8` дал близкое качество tool calling с минимальным числом обучаемых параметров.

Для этого эксперимента я бы выбрал базовую LoRA: она почти сохранила MMLU и улучшила tool EM при меньших затратах памяти, чем у Full FT.

Для сравнения десяти одинаковых запросов сохранён `outputs/manual_review.csv`: по десять ответов исходной модели, Full FT и LoRA. Например, в запросе о конвертации картинки Full FT пропустил `accept: image/png`, а исходная модель и LoRA сохранили его. В запросе о фильмах LoRA вернула эталонный код жанра `878`, Full FT вернул `74`, а исходная модель добавила лишний для exact match аргумент `page=1`.

[Разбор всех десяти примеров](outputs/manual_analysis.csv).
